# 14 · Củng cố B3: phụ thuộc OCR + độ ổn định theo seed (W9)

**Vấn đề:** Kết quả chính W08 (B3 = QLoRA + OCR thắng B2-sub: EM 0,4555 → 0,4925, p = 1,7 × 10⁻⁶) còn hai lỗ hổng mà hội đồng chắc chắn hỏi:
1. **Mới 1 seed (42).** Lợi ích có thể do "may": khởi tạo LoRA, thứ tự batch, dropout.
2. **Chưa biết B3 có phụ thuộc OCR không.** B3 train có dòng OCR; nếu lúc dùng thật OCR hỏng hoặc thiếu thì sao?

**Thiết kế (luật ghi trong `docs/W09/README.md` TRƯỚC khi chạy):**

| Phần | Việc | Thời gian (L4) |
|---|---|---|
| A | Eval adapter **B3 seed 42** trên test2000 **không có dòng OCR** (prompt zero-shot) | ≈ 40 phút |
| B | Train + eval **B2-sub** và **B3** với **seed 43** (tập con train giữ nguyên seed 42) | ≈ 7 giờ |
| B' | Như B với **seed 44** | ≈ 7 giờ |
| C | Gộp 3 seed (42/43/44): mean ± std + kết luận theo luật | 1 phút |

Chỉ đổi **seed huấn luyện**. Tập con 8.000 câu, cấu hình OCR, siêu tham số, GPU **y hệt** W08.

**Chia phiên:** Phần A xong (mục 4) là điểm dừng an toàn. Mỗi seed xong là một điểm dừng. Mở lại notebook, **Run all**: phần đã xong được bỏ qua; train bị ngắt sẽ chạy tiếp từ checkpoint (mất tối đa ≈ 33 phút).

## 1. Thiết lập môi trường

Giống notebook 13 (cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất (có notebook 14), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('notebooks/14_b3_seed_stability_qwen25vl.ipynb'), 'Code cũ -> git pull chưa lấy được commit notebook 14'

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B1/B2/B3). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Tên run + kiểm tra trước khi tốn GPU

- Tập con train đúng **sha256** trong manifest W08 (8.000 câu).
- Cấu hình OCR đọc từ `ocr_selection.json` (W08 bước 5), **không gõ tay**.
- OCR cache đủ **2.670 ảnh train** + **1.486 ảnh test2000**.
- Adapter **B3 seed 42** (W08) còn trên Drive — cần cho Phần A.
- Selftest: `train_qlora` phải in `seed override OK` (có tham số `--seed` mới), `run_baseline`, `seed_stability`.

*Kỳ vọng: `sha256 tập con OK (8000 câu, 2670 ảnh)`, `paddledet_vietocr | v1 | 600 | 0.0`, `train 2670/2670 · test 1486/1486`, `adapter B3 seed 42: OK`, `✅ selftest OK (3 script)`.*

In [ ]:
import json, hashlib, subprocess
from src.data.ocr import default_cache_path, load_ocr_cache, images_of
from src.eval.run_baseline import prepare_images, DATASETS

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vitextvqa_official'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
ADP_ROOT = f'{DRIVE_ROOT}/adapters'
SUB = f'{DRIVE_ROOT}/data/{DS}/train_subset8000_byimage_seed42.json'
TEST = f"{DRIVE_ROOT}/{DATASETS[DS]['data_rel']}"
DIR_A = f'{EXP_ROOT}/W09_B3_ocr_dependency_test2000_L4_seed42'
DIR_C = f'{EXP_ROOT}/W09_B3_seed_stability_test2000_L4'
SAVE_STEPS = 100        # checkpoint ~33 phút/lần (không ảnh hưởng chữ ký run)
SEEDS_NEW = [43, 44]
SEEDS_ALL = [42] + SEEDS_NEW

man = json.load(open('experiments/W08_data_subsets/train_subset8000_byimage_seed42_manifest.json', encoding='utf-8'))
assert hashlib.sha256(open(SUB, 'rb').read()).hexdigest() == man['output_sha256'], 'Tập con train trên Drive khác manifest!'
print(f"sha256 tập con OK ({man['n_questions']} câu, {man['n_images']} ảnh)")

sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'

def run_names(seed):
    # W08 = seed 42 (đã có); W09 = seed mới. Seed trong tên run eval = seed HUẤN LUYỆN của adapter (suy luận greedy).
    w, tail = ('W08' if seed == 42 else 'W09'), f'qwen25vl_vitextvqa_sub8k_ep1_bf16_L4_seed{seed}'
    return {'B2sub':      f'{w}_B2sub_qlora_r16_{tail}',
            'B3':         f'{w}_B3_qlora_ocr_vietocr_v1_r16_{tail}',
            'B2sub_eval': f'{w}_B2sub_eval_qlora_r16_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed{seed}',
            'B3_eval':    f'{w}_B3_eval_qlora_ocr_vietocr_v1_r16_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed{seed}'}
R42 = run_names(42)
B3NOOCR_EVAL = 'W09_B3noocr_eval_qlora_ocr_vietocr_v1_r16_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed42'
for s in SEEDS_ALL:
    print(s, run_names(s)['B3'])

IMG = prepare_images(DS, DRIVE_ROOT)
c = load_ocr_cache(default_cache_path(DRIVE_ROOT, DS, BACKEND))
tr, te = set(images_of([SUB], IMG)), set(images_of([TEST], IMG))
print(f"train {len(tr & set(c['images']))}/{len(tr)} · test {len(te & set(c['images']))}/{len(te)}")
assert tr <= set(c['images']) and te <= set(c['images']), 'OCR cache thiếu ảnh -> chạy notebook 10 Phần B'

AD_B3_42 = f"{ADP_ROOT}/{R42['B3']}/epoch_1"
assert os.path.isdir(AD_B3_42), f'Thiếu adapter {AD_B3_42}'
print('adapter B3 seed 42: OK')

def run_cmd(cmd):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook (không chạy eval trên adapter hỏng)
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}) -> xem log ở trên')

outs = [subprocess.run(f'python -m {m} --selftest', shell=True, capture_output=True, text=True).stdout
        for m in ('src.adaptation.train_qlora', 'src.eval.run_baseline', 'src.eval.seed_stability')]
assert 'seed override OK' in outs[0] and all('SELFTEST OK' in o for o in outs), outs
print('✅ selftest OK (3 script)')

## 3. Phần A — Eval B3 seed 42 **không có OCR** lúc test (≈ 40 phút)

Gắn adapter `epoch_1` của B3 (W08, đã train **có** OCR) nhưng prompt **zero-shot** (không có dòng OCR) — đúng như cách eval B2-sub. Mô phỏng tình huống prototype khi OCR hỏng hoặc không chạy.

*Kỳ vọng: 4 lô, `== KẾT QUẢ: EM=…`. Không có dòng `OCR: …` (vì không dùng OCR).*

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{B3NOOCR_EVAL}/metrics.json'):
    print('⏭ B3 bỏ OCR: đã có -> bỏ qua')
else:
    run_cmd(f'python -u -m src.eval.run_baseline --dataset {DS} --run_name {B3NOOCR_EVAL} --adapter {AD_B3_42}')

## 4. Phần A — Kết quả theo luật

Hai so sánh ghép cặp trên test2000 (luật ghi trước trong README W09):

| So sánh | Câu hỏi | Luật |
|---|---|---|
| B3 (có OCR) → B3 bỏ OCR | B3 có **dựa vào** dòng OCR không? | EM giảm **và** p < 0,05 → *B3 PHỤ THUỘC OCR* |
| B2-sub → B3 bỏ OCR | Train cùng OCR có làm model **yếu đi** khi thiếu OCR không? | EM giảm & p < 0,05 → *KÉM HƠN B2-sub*; EM tăng & p < 0,05 → *VẪN TỐT HƠN B2-sub*; còn lại → *NGANG B2-sub* |

Đây là **chẩn đoán**, không phải kết quả cải tiến.

*Kỳ vọng: bảng 3 dòng, 2 dòng so sánh, 2 dòng `➜ …`, file `ocr_dependency_summary.json`. **Đây là điểm dừng 1** — gửi output mục này cho Claude.*

In [ ]:
import pandas as pd
os.makedirs(DIR_A, exist_ok=True)
def metrics_row(label, run):
    m = json.load(open(f'{EXP_ROOT}/{run}/metrics.json', encoding='utf-8'))
    return {'run': label, 'EM': round(m['metrics']['exact_match'], 4), 'ANLS': round(m['metrics']['anls'], 4),
            'token-F1 (phụ)': round(m['metrics'].get('token_f1', float('nan')), 4), 's/câu': m['meta']['sec_per_sample']}
def paired(run_a, run_b, la, lb, out):
    if not os.path.exists(out):
        r = subprocess.run(f'python src/eval/compare_runs.py --run_a {EXP_ROOT}/{run_a} --run_b {EXP_ROOT}/{run_b} '
                           f'--label_a {la} --label_b {lb} --out {out}', shell=True, capture_output=True, text=True)
        assert r.returncode == 0, r.stdout + r.stderr
    p = json.load(open(out, encoding='utf-8')); p.pop('only_b_question_ids', None)
    print(f"{la:>6} -> {lb:<8} EM {p['em_a']:.4f} -> {p['em_b']:.4f} | chỉ {la} đúng {p['n_only_a']}, chỉ {lb} đúng {p['n_only_b']}, "
          f"p={p['p_value']:.3g} | ΔANLS {p['anls_diff']:+.4f} CI95 [{p['anls_ci95'][0]:+.4f}; {p['anls_ci95'][1]:+.4f}]")
    return p

rows = [metrics_row('B2-sub (không OCR)', R42['B2sub_eval']), metrics_row('B3 (OCR train + test)', R42['B3_eval']),
        metrics_row('B3 bỏ OCR lúc test', B3NOOCR_EVAL)]
display(pd.DataFrame(rows))
p_dep = paired(R42['B3_eval'], B3NOOCR_EVAL, 'B3', 'B3noOCR', f'{DIR_A}/paired_B3_vs_B3noocr.json')
p_harm = paired(R42['B2sub_eval'], B3NOOCR_EVAL, 'B2sub', 'B3noOCR', f'{DIR_A}/paired_B2sub_vs_B3noocr.json')

sig = lambda p: p['p_value'] < 0.05
v_dep = 'B3 PHỤ THUỘC OCR' if (p_dep['em_b'] < p_dep['em_a'] and sig(p_dep)) else 'B3 KHÔNG phụ thuộc OCR có ý nghĩa'
v_harm = ('KÉM HƠN B2-sub' if (p_harm['em_b'] < p_harm['em_a'] and sig(p_harm)) else
          'VẪN TỐT HƠN B2-sub' if (p_harm['em_b'] > p_harm['em_a'] and sig(p_harm)) else 'NGANG B2-sub')
print(f"\n➜ PHỤ THUỘC OCR: {v_dep} (EM {p_dep['em_a']:.4f} -> {p_dep['em_b']:.4f}, p={p_dep['p_value']:.3g})")
print(f"➜ B3 bỏ OCR so với B2-sub: {v_harm} (EM {p_harm['em_a']:.4f} -> {p_harm['em_b']:.4f}, p={p_harm['p_value']:.3g})")
json.dump({'what': 'B3 seed 42 eval KHÔNG có dòng OCR lúc test (W09 Phần A) — chẩn đoán phụ thuộc OCR',
           'rule': 'docs/W09/README.md mục Luật Phần A', 'ocr_config_of_B3': sel, 'table': rows,
           'paired': {'B3_vs_B3noocr': p_dep, 'B2sub_vs_B3noocr': p_harm},
           'verdict_dependency': v_dep, 'verdict_vs_B2sub': v_harm},
          open(f'{DIR_A}/ocr_dependency_summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('->', f'{DIR_A}/ocr_dependency_summary.json')

## 5. Phần B — một seed = 4 bước (định nghĩa hàm, chưa chạy GPU)

Với mỗi seed: train B2-sub → eval B2-sub → train B3 → eval B3. Lệnh **y hệt** notebook 13, chỉ thêm `--seed`. Bước nào đã xong thì bỏ qua. Sau mỗi lần train, kiểm `train_log.json` ghi đúng seed và có adapter `epoch_1` trước khi eval.

*Kỳ vọng: `✅ hàm run_seed sẵn sàng`.*

In [ ]:
def run_seed(seed):
    R = run_names(seed)
    for kind, extra in (('B2sub', ''), ('B3', OCR_ARGS)):
        if os.path.exists(f"{EXP_ROOT}/{R[kind]}/train_log.json"):
            print(f'⏭ {kind} train seed {seed}: đã có -> bỏ qua')
        else:
            run_cmd(f"python -u -m src.adaptation.train_qlora --dataset {DS} --data_file {SUB} --run_name {R[kind]} "
                    f"--epochs 1 --save_steps {SAVE_STEPS} --seed {seed} {extra}")
        meta = json.load(open(f"{EXP_ROOT}/{R[kind]}/train_log.json", encoding='utf-8'))['meta']
        assert meta['seed'] == seed, f"train_log ghi seed {meta['seed']} ≠ {seed}"
        ad = f"{ADP_ROOT}/{R[kind]}/epoch_1"
        assert os.path.isdir(ad), f'Thiếu adapter {ad}'
        ev = R[f'{kind}_eval']
        if os.path.exists(f'{EXP_ROOT}/{ev}/metrics.json'):
            print(f'⏭ {kind} eval seed {seed}: đã có -> bỏ qua')
        else:
            run_cmd(f'python -u -m src.eval.run_baseline --dataset {DS} --run_name {ev} --adapter {ad} {extra}')
        m = json.load(open(f'{EXP_ROOT}/{ev}/metrics.json', encoding='utf-8'))['metrics']
        print(f"✔ {kind} seed {seed}: EM={m['exact_match']:.4f} ANLS={m['anls']:.4f}\n")
print('✅ hàm run_seed sẵn sàng')

## 6. Phần B — Seed 43 (≈ 7 giờ)

B2-sub train ≈ 2,8 giờ → eval ≈ 35 phút → B3 train ≈ 3 giờ → eval ≈ 35 phút. **Giữ máy không ngủ và tab mở.** Bị ngắt thì Run all lại.

*Kỳ vọng: 2 dòng `== XONG: 500 step …` (B2-sub, B3) có `seed=43` ở dòng đầu mỗi run, 2 dòng `== KẾT QUẢ`, 2 dòng `✔ … seed 43`. **Điểm dừng 2.***

In [ ]:
run_seed(43)

## 7. Phần B' — Seed 44 (≈ 7 giờ)

Như mục 6 với seed 44.

*Kỳ vọng: như mục 6 nhưng `seed 44`. **Điểm dừng 3.***

In [ ]:
run_seed(44)

## 8. Phần C — Gộp các seed và kết luận theo luật

Với mỗi seed có đủ 2 eval: so sánh ghép cặp B2-sub vs B3 (`compare_runs.py`), rồi `seed_stability.py` gộp và áp luật:

- **CẢI THIỆN ỔN ĐỊNH:** cả 3 seed EM(B3) > EM(B2-sub) **và** p < 0,05.
- **HƯỚNG NHẤT QUÁN, CHƯA ỔN ĐỊNH:** cả 3 seed cùng chiều nhưng có seed p ≥ 0,05.
- **KHÔNG ỔN ĐỊNH:** có seed EM(B3) ≤ EM(B2-sub).
- **CHƯA ĐỦ SEED:** chạy mục này sớm (vd sau seed 43) chỉ để xem bảng tạm.

Seed 42 tính lại từ predictions W08 và phải khớp `b3_vs_b2sub_summary.json` của W08.

*Kỳ vọng: bảng từng seed, 2 dòng mean ± std, dòng `➜ KẾT LUẬN THEO LUẬT …`, file `seed_stability_summary.json`.*

In [ ]:
os.makedirs(DIR_C, exist_ok=True)
args = []
for s in SEEDS_ALL:
    R = run_names(s)
    if not all(os.path.exists(f"{EXP_ROOT}/{R[k]}/metrics.json") for k in ('B2sub_eval', 'B3_eval')):
        print(f'… seed {s}: chưa đủ 2 eval -> chưa gộp')
        continue
    out = f'{DIR_C}/paired_B2sub_vs_B3_seed{s}.json'
    p = paired(R['B2sub_eval'], R['B3_eval'], 'B2sub', 'B3', out)
    if s == 42:   # kiểm tái lập: phải trùng W08
        w08 = json.load(open('experiments/W08_B3_vs_B2sub_test2000_L4_seed42/paired_B2sub_vs_B3.json', encoding='utf-8'))
        assert (p['n_only_a'], p['n_only_b']) == (w08['n_only_a'], w08['n_only_b']), 'Seed 42 không khớp W08!'
    args.append(f'{s}={out}')
run_cmd(f"python -m src.eval.seed_stability --paired {' '.join(args)} --required_seeds {' '.join(map(str, SEEDS_ALL))} "
        f"--out {DIR_C}/seed_stability_summary.json")

cost = []
for s in SEEDS_ALL:
    for kind in ('B2sub', 'B3'):
        f = f"{EXP_ROOT}/{run_names(s)[kind]}/train_log.json"
        if os.path.exists(f):
            meta = json.load(open(f, encoding='utf-8'))['meta']
            cost.append({'seed': s, 'run': kind, 'giờ train': round(meta['cost']['train_runtime_sec'] / 3600, 2),
                         's/step': meta['cost']['sec_per_step'], 'VRAM đỉnh (GB)': meta['cost']['peak_vram_gb'],
                         'loss cuối': meta['cost']['final_loss'], 'chạy tiếp từ': meta['resumed_from']})
display(pd.DataFrame(cost))

## 9. ⏸ DỪNG

Gửi Claude: output **mục 4** (sau Phần A), và khi xong cả hai seed thì output **mục 8** + các dòng `== XONG` của mục 6 và 7. Sau đó **Runtime → Disconnect and delete runtime**.